In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
import re

# Set seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

2025-11-20 12:02:06.634342: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-20 12:02:08.207363: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-20 12:02:11.297766: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
# Load the dataset
df = pd.read_csv('PoetryFoundationData.csv')

# Display first few rows to understand structure
print(df[['Title', 'Poem']].head())

# Check total number of poems
print(f"\nTotal poems in dataset: {len(df)}")

                                               Title  \
0  \r\r\n                    Objects Used to Prop...   
1  \r\r\n                    The New Church\r\r\n...   
2  \r\r\n                    Look for Me\r\r\n   ...   
3  \r\r\n                    Wild Life\r\r\n     ...   
4  \r\r\n                    Umbrella\r\r\n      ...   

                                                Poem  
0  \r\r\nDog bone, stapler,\r\r\ncribbage board, ...  
1  \r\r\nThe old cupola glinted above the clouds,...  
2  \r\r\nLook for me under the hood\r\r\nof that ...  
3  \r\r\nBehind the silo, the Mother Rabbit\r\r\n...  
4  \r\r\nWhen I push your button\r\r\nyou fly off...  

Total poems in dataset: 13854


In [3]:
# 1. Select a subset of poems to keep training time reasonable
# We join them into one long string (corpus)
data = df['Poem'][:500].astype(str)
corpus = "\n".join(data)

# 2. Convert to lowercase and simple cleaning
corpus = corpus.lower()
# Remove excessive whitespace/newlines for cleaner tokens
corpus = re.sub(r'\r\n', '\n', corpus)
corpus = re.sub(r'\n+', '\n', corpus)

# 3. Tokenization
# Convert words to unique integers
tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])
total_words = len(tokenizer.word_index) + 1

print(f"Total unique words (Vocabulary Size): {total_words}")

Total unique words (Vocabulary Size): 19121


In [4]:
# Create input sequences using list of tokens
input_sequences = []
for line in corpus.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        # Create n-gram sequences
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences so they are all the same length
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Create predictors (X) and label (y)
X, y = input_sequences[:,:-1], input_sequences[:,-1]

# One-hot encode the labels
# Note: This creates a large matrix. If you run out of RAM, reduce the number of poems in Step 3.
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

Shape of X: (108261, 450)
Shape of y: (108261, 19121)


In [5]:
model = Sequential()

# 1. Embedding Layer: Converts integer tokens to dense vectors
# input_dim = vocab size, output_dim = 100, input_length = sequence length - 1 (because label is removed)
model.add(Embedding(input_dim=total_words, output_dim=100, input_length=max_sequence_len-1))

# 2. LSTM Layer: 100 units to capture sequence patterns
model.add(LSTM(100))

# 3. Dropout: To prevent overfitting
model.add(Dropout(0.2))

# 4. Output Layer: Predicts probability of the next word (Softmax)
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.build(input_shape=(None, max_sequence_len-1))
model.summary()

/home/chloycosta/Documents/College_code/Sem_5/NNDL/.venv/lib64/python3.11/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2025-11-20 12:02:18.081654: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 450, 100)       │     1,912,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 100)            │        80,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 19121)          │     1,931,221 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,923,721 (14.97 MB)

 Trainable params: 3,923,721 (14.97 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train for 20 epochs (you can increase this to 50-100 for better results)
history = model.fit(X, y, epochs=30, verbose=1)

2025-11-20 12:02:38.035990: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 194869800 exceeds 10% of free system memory.


In [ ]:
def generate_poem(seed_text, next_words, model, max_sequence_len):
    for _ in range(next_words):
        # Convert seed text to sequence
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        # Pad sequence
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        # Predict next word probabilities
        predicted_probs = model.predict(token_list, verbose=0)
        # Get the index of the highest probability
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]

        # Convert index back to word
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break
        
        # Append to text
        seed_text += " " + output_word
    return seed_text

In [ ]:
print("--- Generated Poetry ---\n")

print(generate_poem("the moon is", 10, model, max_sequence_len))
print(generate_poem("love is like", 10, model, max_sequence_len))
print(generate_poem("darkness falls", 10, model, max_sequence_len))
print(generate_poem("i walked alone", 15, model, max_sequence_len))